In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
import math

def flash_attention(q,k,v,kv_block_size=2):

    n_q, d = q.shape
    n_k = k.shape[0]
    scale = 1.0/math.sqrt(d)

    running_max = q.new_full((n_q,1),-torch.inf)
    running_sum = q.new_zeros(n_q,1)
    accumalator = q.new_zeros(n_q,v.shape[1])

    for start in range(0,n_k,kv_block_size):
        k_tile = k[start:start+kv_block_size]
        v_tile = v[start:start+kv_block_size]

        scores = q @ k.T + scale

        new_max = torch.maximum(running_max,scores.max(-1,keepdim=True).values)
        old_scale = torch.exp(running_max - new_max)
        tile_weights = torch.exp(scores-new_max)
        running_sum = old_scale * running_sum + tile_weights.sum(-1,keepdim=True)
        accumalator = old_scale * accumalator + tile_weights @ v_tile
        running_max = new_max

    return accumalator / running_sum

In [ ]:
def flash_attention(q,k,v,kv_block_size=2):

    n_q, d = q.shape
    n_k = k.shape[0]

    scale = 1.0/ math.sqrt(d)

    running_max = q.new_full((n_q,1),-torch.inf)
    running_sum = q.new_zeros(n_q,1)
    accumlator = q.new_zeros(n_q,v.shape[0])

    for start in range(0,n_k,kv_block_size):

        k_tile = k[start:start+kv_block_size]
        v_tile = v[start:start+kv_block_size]

        scores = ( q @ k_tile.T)*scale

        new_max = torch.maximum(
            running_max,
            scores.max(-1,keepdim=True).values
        )

        old_scale = torch.exp(running_max-new_max)

        tile_weights = torch.exp(scores-new_max)

        running_sum = (
            old_scale * running_sum
            + tile_weights.sum(-1,keepdim=True)
        )

        accumlator = (
            old_scale * accumlator+ tile_weights @ v_tile
        )

        running_max = new_max

    return accumlator / running_sum